# PCA Model
## Marissa Burton

In [61]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA, TruncatedSVD as SVD
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'notebook'

In [62]:
TFIDF_raw = pd.read_csv('data/TFIDF.csv').set_index('book_id')
TFIDF_L2 = pd.read_csv('data/TFIDF_L2.csv').set_index('book_id')
LIB = pd.read_csv('data/LIB.csv')

### DCM creation

In [63]:
pca_engine = PCA(n_components=5)
DCM = pd.DataFrame(pca_engine.fit_transform(TFIDF_L2), index=TFIDF_L2.index)
DCM.columns = ['PC{}'.format(i) for i in DCM.columns]
DCM

,PC0,PC1,PC2,PC3,PC4
book_id,,,,,
43,0.143717,-0.184738,-0.305083,-0.063710,0.759035
175,0.138283,-0.218892,-0.263401,0.009611,-0.383000
345,0.129366,-0.131710,-0.302139,-0.005749,-0.079064
696,-0.314267,0.127297,0.295571,-0.433152,-0.056275
1952,0.504837,0.800591,0.010125,-0.018371,0.030088
3268,-0.326245,0.004679,0.002116,0.006884,-0.094086
5182,-0.311846,0.070521,0.119373,-0.335772,0.034473
6087,-0.236393,0.120801,0.173924,0.736603,0.030445
12122,0.495924,-0.394071,0.689166,0.027394,0.051532


### LOADINGS creation

In [64]:
COMPS = pd.DataFrame(pca_engine.components_.T * np.sqrt(pca_engine.explained_variance_))
COMPS.columns = ["PC{}".format(i) for i in COMPS.columns]
COMPS.index = TFIDF_L2.columns
COMPS.index.name = 'term_str'
LOADINGS = COMPS.T

### Top 5 Positive Terms for First Component

In [65]:
positive = COMPS.sort_values('PC0', ascending=False).head(5).reset_index()
positive['term_str'].tolist()

['herbert', 'paw', 't', 'eleanor', 'don']

### Top 5 Negative Terms for Second Component

In [66]:
negative = COMPS.sort_values('PC1', ascending=True).head(5).reset_index()
negative['term_str'].tolist()

['herbert', 'paw', 'sergeantmajor', 'talisman', 'monkey']

### Visualizations

In [73]:
def vis_pcs(DCM, a, b, label, hover_name):
    return px.scatter(DCM, f"PC{a}", f"PC{b}", 
                    color=LIB[label], 
                    hover_name=LIB[hover_name], 
                    size=LIB['book_len'],
                    marginal_x='box', height=800)

In [68]:
def vis_loadings(COMPS, a=0, b=1, hover_name='term_str'):
    return px.scatter(COMPS.reset_index(), f"PC{a}", f"PC{b}", 
                      text='term_str', 
                      marginal_x='box', 
                      height=800)

#### First 2 Components (0 and 1)

In [76]:
vis_pcs(DCM, 0, 1, 'era', 'title')

In [70]:
vis_loadings(COMPS, 0, 1)

#### Components 1 and 2

In [77]:
vis_pcs(DCM, 1, 2, 'era', 'title')

In [78]:
vis_loadings(COMPS, 1, 2)

### Export tables

In [71]:
DCM.to_csv('data/DCM.csv')

In [72]:
LOADINGS.to_csv('data/LOADINGS.csv')